In [2]:
!pip install pandas nltk

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


<h3 style="color:blue;">ÉTAPE 2 -Import et téléchargement des ressources NLTK</h3> 

Avant de coder, NLTK a besoin de télécharger des "paquets" spécifiques :

   - **punkt & punkt_tab :** Pour diviser le texte en mots (tokenisation).

   - **stopwords :** La liste des mots fréquents mais peu informatifs (the, is, in, etc.).

   - **wordnet :** Le dictionnaire utilisé pour ramener un mot à sa racine (lemmatisation).

In [3]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /home/fateh/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /home/fateh/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /home/fateh/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /home/fateh/nltk_data...


True

<h3 style="color:blue;">ÉTAPE 3 — Chargement de la base de données</h3> 

On importe le fichier data.csv. L'affichage des 20 premières lignes permet de valider le bon formatage des colonnes (id, type, domain, contenu) et de repérer d'éventuels problèmes de caractères spéciaux.

In [5]:
import pandas as pd

df = pd.read_csv("../Data/data.csv")

df.head(20)

,id,type,domain,contenu
0,1,medical,Pharmacolo,8279 Neurosci Lett 2003 Jun 12 343 3 205-9 doi...
1,2,non_medical,Cinéma,Sunny Wayne Sujith Unnikrishnan born 19 August...
2,3,non_medical,Automobile,J J Deal and Son Carriage Factory The J J Deal...
3,4,non_medical,Travel,The Amazing Race 27 The Amazing Race 27 is th...
4,5,medical,Cardiologi,6038 Am J Cardiol 1994 Mar 1 73 7 524-5 doi 10...
5,6,medical,Vaccinatio,4510 Pan Afr Med J 2022 Feb 8 41 112 doi 10 11...
6,7,medical,biologie,3354 J Nat Prod 1980 Nov 43 6 721-3 doi 10 102...
7,8,non_medical,Education,Education system in Kuwait Kuwait has an exte...
8,9,medical,Mdicaments,5119 Zhonghua Yi Shi Za Zhi 2009 May 39 3 168-...
9,10,medical,Cardiologi,190 J Cardiovasc Med Hagerstown 2018 May 19 5 ...


<h3 style="color:blue;">ÉTAPE 4 — Nettoyage de base du texte</h3> 

Ici, on uniformise le texte pour que l'ordinateur ne soit pas perturbé par des détails inutiles :

    
   - **Minuscules :** Pour que "Sante" et "sante" soient vus comme le même mot.

   -  **Suppression des accents :** On normalise les caractères (ex: "é" devient "e").

   - **Suppression des chiffres et de la ponctuation :** On ne garde que les lettres pour se concentrer sur le sens sémantique.

In [6]:
import re
import unicodedata

def clean_text_basic(text):
    text = str(text)

    # 🔹 minuscule
    text = text.lower()

    # 🔹 suppression accents
    text = unicodedata.normalize('NFD', text)
    text = text.encode('ascii', 'ignore').decode('utf-8')

    # 🔹 suppression chiffres
    text = re.sub(r"\d+", "", text)

    # 🔹 suppression ponctuation
    text = re.sub(r"[^\w\s]", "", text)

    return text

<h3 style="color:blue;">ÉTAPE 5 — Tokenisation</h3> 

Le texte est découpé en unités individuelles appelées tokens.

  Exemple : **"Le chat dort"** devient **["Le", "chat", "dort"]**.

  Cela permet d'appliquer des filtres sur chaque mot séparément.

In [7]:
from nltk.tokenize import word_tokenize

def tokenize(text):
    return word_tokenize(text)

<h3 style="color:blue;">ÉTAPE 6 — Suppression des stopwords</h3> 

Dans cette étape, on supprime les mots inutiles (stopwords) comme (the, is ,and....)

Ces mots n’apportent pas d’information utile pour la classification.

In [8]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

<h3 style="color:blue;">ÉTAPE 7 — Lemmatisation</h3> 

Dans cette étape, on transforme les mots en leur forme de base.

Exemples :

- running → run
- studies,studying → study

Cela permet de normaliser les mots.

In [9]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

<h3 style="color:blue;">ÉTAPE 8 — Pipeline complet de prétraitement</h3> 

On crée une fonction "maître" (preprocess) qui enchaîne toutes les étapes précédentes dans l'ordre. C'est le centre de contrôle du nettoyage de chaque document.

In [10]:
def preprocess(text):
    text = clean_text_basic(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize(tokens)
    return " ".join(tokens)

<h3 style="color:blue;">ÉTAPE 9 — Application sur le dataset</h3> 

C'est ici que le traitement lourd commence :

   - On applique la fonction preprocess sur chaque ligne de la colonne contenu.

   - On crée une nouvelle colonne clean_text pour stocker le résultat final.

   - Cela nous permet de comparer visuellement le texte brut et le texte "propre".

In [11]:
df["clean_text"] = df["contenu"].apply(preprocess)

df.head(20)

,id,type,domain,contenu,clean_text
0,1,medical,Pharmacolo,8279 Neurosci Lett 2003 Jun 12 343 3 205-9 doi...,neurosci lett jun doi distinct effect amantadi...
1,2,non_medical,Cinéma,Sunny Wayne Sujith Unnikrishnan born 19 August...,sunny wayne sujith unnikrishnan born august kn...
2,3,non_medical,Automobile,J J Deal and Son Carriage Factory The J J Deal...,j j deal son carriage factory j j deal son car...
3,4,non_medical,Travel,The Amazing Race 27 The Amazing Race 27 is th...,amazing race amazing race twentyseventh season...
4,5,medical,Cardiologi,6038 Am J Cardiol 1994 Mar 1 73 7 524-5 doi 10...,j cardiol mar doi memoriam herman k hellerstei...
5,6,medical,Vaccinatio,4510 Pan Afr Med J 2022 Feb 8 41 112 doi 10 11...,pan afr med j feb doi pamj ecollection prevale...
6,7,medical,biologie,3354 J Nat Prod 1980 Nov 43 6 721-3 doi 10 102...,j nat prod nov doi npa valepotriates tissue cu...
7,8,non_medical,Education,Education system in Kuwait Kuwait has an exte...,education system kuwait kuwait extensive educa...
8,9,medical,Mdicaments,5119 Zhonghua Yi Shi Za Zhi 2009 May 39 3 168-...,zhonghua yi shi za zhi may elementary explorat...
9,10,medical,Cardiologi,190 J Cardiovasc Med Hagerstown 2018 May 19 5 ...,j cardiovasc med hagerstown may doi jcm cardio...


<h3 style="color:blue;">ÉTAPE 10 — Qualité des données (Valeurs nulles et doublons)</h3> 

Pour garantir un modèle de Machine Learning performant, on nettoie la structure du tableau :

   - **dropna() :** On supprime les lignes vides.

   - **drop_duplicates() :** On élimine les doublons exacts pour ne pas biaiser les futures statistiques.

In [12]:
df = df.dropna()
df = df.drop_duplicates(subset=["clean_text"])

df_clean = df[["id", "type", "domain", "clean_text"]]

<h3 style="color:blue;">ÉTAPE 11 — Sauvegarde du dataset propre</h3> 
On sauvegarde le résultat dans un nouveau fichier CSV (clean_dataset.csv). Ce fichier est prêt à être utilisé pour l'entraînement d'une intelligence artificielle ou pour faire de l'analyse statistique textuelle.

In [13]:
df.to_csv("../Data/clean_dataset.csv", index=False)